> ## SOLUTION / ANSWER KEY &mdash; Lab 8.7
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-07-voting.ipynb`](../lab-07-voting.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 8.7 &mdash; Voting & Consensus

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Take the majority answer from several agents
- Require a clear majority above a threshold
- Escalate a split (no majority) to a human

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it and read what happened**. The multi-agent *mechanics* (routing, shared state, voting, critique, synthesis, observability) are **real Python you build and run**; the **specialist agents and the assembled chatbot are real `create_agent` agents** that really answer. Finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is a working team and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langgraph`) and, for the specialists, a **real hosted model** &mdash; `ChatGroq(model="openai/gpt-oss-20b")` with your `GROQ_API_KEY` from `.env`. If the key is missing, the live cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; Voting & consensus](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"), override=True)   # GROQ_API_KEY (Day-4 provider)

WORK = "/tmp/biaa-lab-08-07"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is present. The live specialist cells self-skip when it is absent."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
# gpt-oss-20b is verified; do NOT use llama-3.3-70b-versatile (it 400s through create_agent).
# One shared model is fine -- each specialist differs by its system_prompt + its own tools.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY loaded | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
When agents produce **comparable answers**, take a **vote** (deck slide 12). Independent errors cancel: if
each agent is 70% right with uncorrelated mistakes, the **majority** is far better. But a **split vote** is a
*signal* &mdash; escalate to a human rather than force a call. (And beware a **shared blind spot**: agents
that reason alike will confidently agree on the same wrong answer.)

In [ ]:
from collections import Counter
print("three agents voted:", ["refund", "refund", "deny"], "-> majority?")

## Build it
Complete `vote` (the majority) and `decide` (escalate unless a clear majority).

In [ ]:
from collections import Counter

def vote(answers):
    counts = Counter(answers)
    top, n = counts.most_common(1)[0]
    return top

def decide(answers, threshold=0.5):
    counts = Counter(answers)
    top, n = counts.most_common(1)[0]
    if n / len(answers) > threshold:
        return {"decision": top, "escalate": False}
    return {"decision": None, "escalate": True}   # split -> escalate to a human

try:
    print("vote  :", vote(["refund", "refund", "deny"]))
    print("clear :", decide(["refund", "refund", "deny"]))
    print("split :", decide(["refund", "deny", "wait"]))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- `vote` returns the **majority** answer; a bare 2/3 majority clears the default `threshold` and is decided.
- A three-way **split** returns `escalate: True` &mdash; the system *knows it doesn't know* and hands off to a human.
- This same `decide` is wired into the capstone (Lab 8.12) to resolve a **disputed refund** &mdash; a real place a split must escalate.

## Your turn (open task &mdash; no grader)
Raise the `threshold` to `0.66` and re-run over `["refund", "refund", "deny"]`. **What good looks like:** a
2/3 majority now *fails* the stricter bar and escalates &mdash; showing the threshold is your dial between
*trust the majority* and *only act on strong consensus*. Pick the bar to match how costly a wrong call is.

---
### Key takeaway
Voting converges comparable answers, and a split vote is information -- escalate, don't force it. Use it for checkable answers; watch for a shared blind spot that makes agents agree on the same mistake.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>